In [ ]:
# Mount Drive on second account

from google.colab import drive
drive.mount('/content/drive')

import os

# try to find the shared folder
possible_paths = [
    '/content/drive/MyDrive/HeyCareLog_Dataset',
    '/content/drive/Shareddrives/HeyCareLog_Dataset',
]

BASE = None
for path in possible_paths:
    if os.path.exists(path):
        BASE = path
        break

# if still not found search for it
if BASE is None:
    for root, dirs, files in os.walk('/content/drive'):
        for d in dirs:
            if 'HeyCareLog' in d:
                BASE = os.path.join(root, d)
                break
        if BASE:
            break

print(f'BASE = {BASE}')
print('Contents:', os.listdir(BASE))

os.makedirs(f'{BASE}/models/disfluency/bart_v2', exist_ok=True)
os.makedirs(f'{BASE}/results', exist_ok=True)
print('Ready!')

Mounted at /content/drive
BASE = /content/drive/MyDrive/HeyCareLog_Dataset
Contents: ['audio', 'labels', 'podcastfillers', 'models', 'results', 'notebooks']
Ready!


Install Libraries

In [ ]:
# Install libraries

!pip install -q transformers
!pip install -q datasets
!pip install -q accelerate
!pip install -q evaluate
!pip install -q sentencepiece
!pip install -q rouge_score

import torch
print('Libraries installed!')
print(f'GPU available: {torch.cuda.is_available()}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Libraries installed!
GPU available: True


 Load Data

In [ ]:
# Load training data

import pandas as pd

train_df = pd.read_csv(f'{BASE}/labels/train_text.csv')
val_df   = pd.read_csv(f'{BASE}/labels/val_text.csv')
test_df  = pd.read_csv(f'{BASE}/labels/test_text.csv')

print(f'Train: {len(train_df)} rows')
print(f'Val:   {len(val_df)} rows')
print(f'Test:  {len(test_df)} rows')

print('\nDisfluency types:')
print(train_df['auto_edit_features'].value_counts())

Train: 967 rows
Val:   121 rows
Test:  121 rows

Disfluency types:
auto_edit_features
filler_word_removal; spoken_self_correction_handling                        540
filler_word_removal                                                         280
filler_word_removal; spoken_self_correction_handling; repetition_removal     94
filler_word_removal; repetition_removal                                      53
Name: count, dtype: int64


Load Metrics

In [ ]:
# Load evaluation metrics

import evaluate

bleu_metric  = evaluate.load('bleu')
rouge_metric = evaluate.load('rouge')

def compute_scores(predictions, references):
    bleu = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references]
    )['bleu']
    rouge = rouge_metric.compute(
        predictions=predictions,
        references=references
    )['rougeL']
    return {
        'BLEU':    round(bleu,  4),
        'ROUGE-L': round(rouge, 4)
    }

print('Metrics ready!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Metrics ready!


METHOD 1: Data Augmentation

In [ ]:
# METHOD 1: DATA AUGMENTATION

# Research support:
#   Shorten & Khoshgoftaar (2019) — "A survey on Image Data Augmentation
#   for Deep Learning" — proves augmentation reduces overfitting
#   when training data lacks diversity.

aug_noisy = []
aug_clean = []

# keep all original 968 pairs
for _, row in train_df.iterrows():
    aug_noisy.append(str(row['speech_to_text_output']))
    aug_clean.append(str(row['expected_cleaned_text']))

print(f'Original pairs: {len(aug_noisy)}')

#  Augmentation 1: Double fillers
# teaches model that 'uh uh' and 'um um' are also fillers
# your original data has single 'uh' and 'um' only
for _, row in train_df.iterrows():
    noisy = str(row['speech_to_text_output'])
    clean = str(row['expected_cleaned_text'])
    doubled = noisy.replace(' uh ', ' uh uh ')
    doubled = doubled.replace(' um ', ' um um ')
    if doubled != noisy:
        aug_noisy.append(doubled)
        aug_clean.append(clean)

print(f'After double filler augmentation: {len(aug_noisy)}')

# Augmentation 2: New self-correction patterns
# teaches model to resolve 'X no sorry Y' → keep only Y
# these are brand new pairs not in your original data
self_correction_pairs = [
    ("Medicine given one tablet no sorry two tablets as instructed",
     "Medicine given two tablets as instructed."),
    ("She walked for 10 minutes no sorry 15 minutes today",
     "She walked for 15 minutes today."),
    ("Diaper changed uh two times actually three times today",
     "Diaper changed three times today."),
    ("Fluid intake was 200 ml no sorry 300 ml this morning",
     "Fluid intake was 300 ml this morning."),
    ("Breakfast was given at 9 AM no sorry 10 AM",
     "Breakfast was given at 10 AM."),
    ("She ate half of the meal actually full meal today",
     "She ate full meal today."),
    ("Medicine was refused no sorry was given after dinner",
     "Medicine was given after dinner."),
    ("Patient walked with support no sorry without support",
     "Patient walked without support."),
    ("She had partial bath no sorry full bath this morning",
     "She had a full bath this morning."),
    ("Mood was confused no sorry calm this afternoon",
     "Mood was calm this afternoon."),
    ("She drank 100 ml water no sorry 200 ml water today",
     "She drank 200 ml water today."),
    ("Medicine after breakfast uh one tablet no sorry two tablets",
     "Medicine after breakfast, two tablets."),
    ("Lunch was given at 12 PM no sorry 1 PM today",
     "Lunch was given at 1 PM today."),
    ("She had rice no sorry bread for breakfast this morning",
     "She had bread for breakfast this morning."),
    ("Patient was calm no sorry agitated this evening",
     "Patient was agitated this evening."),
    ("Diaper changed once no sorry twice in the morning",
     "Diaper changed twice in the morning."),
    ("She drank uh 150 ml actually 250 ml of water",
     "She drank 250 ml of water."),
    ("Medicine dose was one tablet no sorry half tablet",
     "Medicine dose was half tablet."),
    ("She had uh soup no sorry rice for dinner tonight",
     "She had rice for dinner tonight."),
    ("Patient slept for 6 hours no sorry 8 hours last night",
     "Patient slept for 8 hours last night."),
]

for noisy, clean in self_correction_pairs:
    aug_noisy.append(noisy)
    aug_clean.append(clean)
    aug_noisy.append('uh ' + noisy)
    aug_clean.append(clean)

print(f'After self-correction augmentation: {len(aug_noisy)}')

# Augmentation 3: Repetition patterns
# teaches model to detect and remove repeated phrases
repetition_pairs = [
    ("Diaper was changed diaper was changed 2 times today mood was calm",
     "Diaper was changed 2 times today. Mood was calm."),
    ("Medicine was given medicine was given after breakfast today",
     "Medicine was given after breakfast today."),
    ("Breakfast was given breakfast was given at 10 AM this morning",
     "Breakfast was given at 10 AM this morning."),
    ("She had she had a full bath this morning",
     "She had a full bath this morning."),
    ("Mood was calm mood was calm throughout the day",
     "Mood was calm throughout the day."),
    ("Patient walked patient walked with support today",
     "Patient walked with support today."),
    ("Lunch was given lunch was given at 1 PM",
     "Lunch was given at 1 PM."),
    ("She ate half she ate half of the rice for lunch",
     "She ate half of the rice for lunch."),
    ("Diaper changed diaper changed 3 times today",
     "Diaper changed 3 times today."),
    ("Medicine refused medicine refused after dinner",
     "Medicine refused after dinner."),
    ("Body pain body pain was observed in the evening",
     "Body pain was observed in the evening."),
    ("Fluid intake fluid intake was 450 ml today",
     "Fluid intake was 450 ml today."),
    ("She had she had rice for lunch today",
     "She had rice for lunch today."),
    ("Patient was patient was confused this morning",
     "Patient was confused this morning."),
    ("Wheelchair movement wheelchair movement with caregiver support",
     "Wheelchair movement with caregiver support."),
]

for noisy, clean in repetition_pairs:
    aug_noisy.append(noisy)
    aug_clean.append(clean)
    aug_noisy.append('uh ' + noisy)
    aug_clean.append(clean)

print(f'After repetition augmentation: {len(aug_noisy)}')

# Augmentation 4: Mixed filler positions
# teaches model fillers can appear anywhere in a sentence
mixed_pairs = [
    ("She had um a partial body wash this morning",
     "She had a partial body wash this morning."),
    ("Medicine was uh given after breakfast today",
     "Medicine was given after breakfast today."),
    ("Mood was um confused throughout the day",
     "Mood was confused throughout the day."),
    ("She uh ate full lunch and drank 250 ml water",
     "She ate full lunch and drank 250 ml water."),
    ("Diaper was uh changed 3 times today",
     "Diaper was changed 3 times today."),
    ("Patient had um wheelchair movement today",
     "Patient had wheelchair movement today."),
    ("She um drank about 300 ml of water today",
     "She drank about 300 ml of water today."),
    ("Breakfast was uh given at 10 AM this morning",
     "Breakfast was given at 10 AM this morning."),
    ("Medicine after dinner was um refused today",
     "Medicine after dinner was refused today."),
    ("She had uh uh full body bath this morning",
     "She had a full body bath this morning."),
    ("Patient was um um calm throughout the day",
     "Patient was calm throughout the day."),
    ("Diaper changed um um 2 times today",
     "Diaper changed 2 times today."),
    ("She uh had plain tea at um 4 PM today",
     "She had plain tea at 4 PM today."),
    ("Patient um walked with support for 10 minutes",
     "Patient walked with support for 10 minutes."),
    ("Symptoms observed uh body pain this evening",
     "Symptoms observed: body pain this evening."),
]

for noisy, clean in mixed_pairs:
    aug_noisy.append(noisy)
    aug_clean.append(clean)

print(f'Final augmented dataset: {len(aug_noisy)} pairs')
print(f'New examples added: {len(aug_noisy) - len(train_df)}')

Original pairs: 967
After double filler augmentation: 1934
After self-correction augmentation: 1974
After repetition augmentation: 2004
Final augmented dataset: 2019 pairs
New examples added: 1052


METHOD 2 + 3: Re-Train with Reduced LR + Weight Decay

In [ ]:
# RE-TRAIN BART WITH ALL 3 FIXES
#
# METHOD 2: REDUCED LEARNING RATE
#   Was: 5e-5
#   Now: 2e-5 (2.5x smaller)
#   Why: Smaller LR = model learns more carefully
#        Less likely to memorise specific patterns
#
# METHOD 3: WEIGHT DECAY (L2 REGULARISATION)
#   weight_decay = 0.01
#   Why: Adds mathematical penalty for memorising
#        Forces model to find simpler general solutions
#   Research support:
#     Loshchilov & Hutter (2019) — Decoupled Weight Decay
#     Regularisation — ICLR 2019
#

from transformers import (
    BartTokenizer, BartForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from datasets import Dataset
import pandas as pd, gc, torch

# create augmented dataframe
import pandas as pd
aug_df = pd.DataFrame({
    'noisy': aug_noisy,
    'clean': aug_clean
}).drop_duplicates().sample(
    frac=1, random_state=42
).reset_index(drop=True)

print(f'Total augmented pairs: {len(aug_df)}')

# split 90% train 10% val
split     = int(len(aug_df) * 0.9)
aug_train = aug_df[:split]
aug_val   = aug_df[split:]

print(f'Aug train: {len(aug_train)}')
print(f'Aug val:   {len(aug_val)}')

# load fresh BART model from HuggingFace
MODEL_NAME = 'facebook/bart-base'
tokenizer  = BartTokenizer.from_pretrained(MODEL_NAME)
model      = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

MAX_INPUT  = 512
MAX_TARGET = 400

def tokenize_pair(batch):
    model_inputs = tokenizer(
        batch['noisy'],
        max_length=MAX_INPUT,
        truncation=True,
        padding='max_length'
    )
    labels = tokenizer(
        batch['clean'],
        max_length=MAX_TARGET,
        truncation=True,
        padding='max_length'
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_data = Dataset.from_dict({
    'noisy': aug_train['noisy'].tolist(),
    'clean': aug_train['clean'].tolist()
}).map(tokenize_pair, batched=True, batch_size=16)

val_data = Dataset.from_dict({
    'noisy': aug_val['noisy'].tolist(),
    'clean': aug_val['clean'].tolist()
}).map(tokenize_pair, batched=True, batch_size=16)

training_args = Seq2SeqTrainingArguments(
    output_dir=f'{BASE}/models/disfluency/bart_v2',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    # METHOD 2: reduced learning rate
    learning_rate=2e-5,

    warmup_steps=200,

    # METHOD 3: weight decay regularisation
    weight_decay=0.01,

    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    predict_with_generate=True,
    generation_max_length=400,
    fp16=True,
    logging_steps=50,
    report_to=['none'],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, model=model, padding=True
    ),
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=1
    )]
)

print()
print('=== RE-TRAINING BART WITH 3 OVERFITTING FIXES ===')
print()
print('Fix 1 — Data Augmentation:')
print(f'  Original: {len(train_df)} pairs')
print(f'  Augmented: {len(aug_train)} pairs')
print()
print('Fix 2 — Reduced Learning Rate:')
print('  Was: 5e-5  →  Now: 2e-5')
print()
print('Fix 3 — Weight Decay:')
print('  weight_decay = 0.01')
print()
print('Expected time: 20-35 minutes')
print()

trainer.train()

print()
print('Re-training complete!')
print(f'Saved to: {BASE}/models/disfluency/bart_v2')

Total augmented pairs: 2019
Aug train: 1817
Aug val:   202


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



Re-training complete!
Saved to: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/bart_v2


Find Best Checkpoint

In [ ]:
#  Find the best saved checkpoint

import os

v2_path = f'{BASE}/models/disfluency/bart_v2'
checkpoints = sorted([
    d for d in os.listdir(v2_path)
    if d.startswith('checkpoint')
])

BART_V2 = f'{v2_path}/{checkpoints[-1]}'
print(f'Best checkpoint: {BART_V2}')
print(f'Files: {os.listdir(BART_V2)}')

Best checkpoint: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/bart_v2/checkpoint-684
Files: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']


Test on 5 Sentences That Failed Before

In [ ]:
# Test BART v2 on sentences that failed before
# These are the exact same 5 sentences from the overfitting check
# Compare results with original BART

from transformers import BartTokenizer, BartForConditionalGeneration
import torch, gc

bart_tok   = BartTokenizer.from_pretrained(BART_V2)
bart_model = BartForConditionalGeneration.from_pretrained(BART_V2)
bart_model.eval()

def clean_bart(noisy):
    toks = bart_tok(
        noisy, return_tensors='pt',
        max_length=512, truncation=True
    )
    with torch.no_grad():
        out = bart_model.generate(
            **toks, max_new_tokens=400, num_beams=4
        )
    return bart_tok.decode(out[0], skip_special_tokens=True)

test_cases = [
    ("Filler removal",
     "She had uh uh rice for lunch and drank um about 300 ml water",
     "She had rice for lunch and drank about 300 ml of water."),

    ("Self-correction",
     "Medicine was given after breakfast uh one tablet no sorry two tablets as the doctor said",
     "Medicine was given after breakfast, two tablets as instructed."),

    ("Repetition removal",
     "Diaper was changed diaper was changed 2 times today mood was calm",
     "Diaper was changed 2 times today. Mood was calm."),

    ("Complex mixed",
     "She had uh full body bath this morning and um breakfast was given she ate half of the rice and drank about 150 ml water medicine after dinner was refused no sorry was given",
     "She had a full body bath this morning. Breakfast was given. She ate half of the rice and drank about 150 ml of water. Medicine after dinner was given."),

    ("Self-correction 2",
     "Patient had uh physiotherapy session today um she walked with support for about 10 minutes no sorry 15 minutes",
     "Patient had a physiotherapy session today. She walked with support for about 15 minutes."),
]

# original BART v1 results for comparison
v1_outputs = [
    "She had uh uh rice for lunch and drank um about 300 ml of water.",
    "Medicine was given after breakfast, one tablet, two tablets as instructed by the doctor.",
    "Diaper was changed diaper was changed 2 times today. Mood was calm.",
    "She had a full body bath this morning and um breakfast was given...",
    "Patient had a physiotherapy session today. She walked with support for about 10 minutes. No sorry 15 minutes.",
]

print('='*70)
print('COMPARISON: BART v1 (original) vs BART v2 (overfitting fixed)')
print('='*70)

improved = 0
for i, (label, noisy, expected) in enumerate(test_cases):
    v2_output = clean_bart(noisy)

    fillers_in_v2 = any(
        w in v2_output.lower()
        for w in [' uh ', ' um ', 'no sorry']
    )
    status = 'Fixed' if not fillers_in_v2 else ' Partial'
    if not fillers_in_v2:
        improved += 1

    print(f'\nTest {i+1}: {label}')
    print(f'  INPUT:    {noisy}')
    print(f'  EXPECTED: {expected}')
    print(f'  V1 OUTPUT:{v1_outputs[i]}')
    print(f'  V2 OUTPUT:{v2_output}')
    print(f'  STATUS:   {status}')

print()
print(f'Improved: {improved}/{len(test_cases)} tests')
print('(compare with V1 which had issues on all 5 tests)')

del bart_model, bart_tok
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

COMPARISON: BART v1 (original) vs BART v2 (overfitting fixed)

Test 1: Filler removal
  INPUT:    She had uh uh rice for lunch and drank um about 300 ml water
  EXPECTED: She had rice for lunch and drank about 300 ml of water.
  V1 OUTPUT:She had uh uh rice for lunch and drank um about 300 ml of water.
  V2 OUTPUT:She had rice for lunch and drank about 300 ml of water.
  STATUS:   Fixed

Test 2: Self-correction
  INPUT:    Medicine was given after breakfast uh one tablet no sorry two tablets as the doctor said
  EXPECTED: Medicine was given after breakfast, two tablets as instructed.
  V1 OUTPUT:Medicine was given after breakfast, one tablet, two tablets as instructed by the doctor.
  V2 OUTPUT:Medicine was given after breakfast, two tablets as instructed by the doctor.
  STATUS:   Fixed

Test 3: Repetition removal
  INPUT:    Diaper was changed diaper was changed 2 times today mood was calm
  EXPECTED: Diaper was changed 2 times today. Mood was calm.
  V1 OUTPUT:Diaper was changed dia

Evaluate on Full Test Set and Compare V1 vs V2

In [ ]:
# Compare BART v1 vs v2 on full 120 test rows

import torch, gc
from transformers import BartTokenizer, BartForConditionalGeneration

BART_V1 = f'{BASE}/models/disfluency/bart/checkpoint-363'

def evaluate_bart(path):
    tok = BartTokenizer.from_pretrained(path)
    mdl = BartForConditionalGeneration.from_pretrained(path)
    mdl.eval()
    preds = []
    refs  = []

    for _, row in test_df.iterrows():
        inp  = str(row['speech_to_text_output'])
        toks = tok(inp, return_tensors='pt',
                   max_length=512, truncation=True)
        with torch.no_grad():
            out = mdl.generate(
                **toks, max_new_tokens=400,
                num_beams=4, early_stopping=True
            )
        preds.append(tok.decode(out[0], skip_special_tokens=True))
        refs.append(str(row['expected_cleaned_text']))

    del mdl, tok
    gc.collect()
    torch.cuda.empty_cache()
    return compute_scores(preds, refs)

print('Evaluating BART v1...')
v1 = evaluate_bart(BART_V1)
print(f'  V1: {v1}')

print('Evaluating BART v2...')
v2 = evaluate_bart(BART_V2)
print(f'  V2: {v2}')

print()
print('='*55)
print('FINAL COMPARISON: V1 vs V2')
print('='*55)
print(f'{"Model":<20} {"BLEU":>8} {"ROUGE-L":>10}')
print('-'*55)
print(f'{"BART v1 original":<20} {v1["BLEU"]:>8.4f} {float(v1["ROUGE-L"]):>10.4f}')
print(f'{"BART v2 fixed":<20} {v2["BLEU"]:>8.4f} {float(v2["ROUGE-L"]):>10.4f}')
print('='*55)

import pandas as pd
pd.DataFrame([
    {'Model':'BART-v1','BLEU':v1['BLEU'],'ROUGE_L':float(v1['ROUGE-L'])},
    {'Model':'BART-v2','BLEU':v2['BLEU'],'ROUGE_L':float(v2['ROUGE-L'])}
]).to_csv(f'{BASE}/results/bart_v1_vs_v2.csv', index=False)

print('Results saved!')

Evaluating BART v1...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

  V1: {'BLEU': 0.8549, 'ROUGE-L': np.float64(0.9339)}
Evaluating BART v2...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

  V2: {'BLEU': 0.8525, 'ROUGE-L': np.float64(0.94)}

FINAL COMPARISON: V1 vs V2
Model                    BLEU    ROUGE-L
-------------------------------------------------------
BART v1 original       0.8549     0.9339
BART v2 fixed          0.8525     0.9400
Results saved!


In [ ]:
# TEST BART V2 - 5 Different Sentence Types
# Comparing V1 vs V2 side by side

from transformers import BartTokenizer, BartForConditionalGeneration
import torch, gc, os

# load BART v2
v2_path     = f'{BASE}/models/disfluency/bart_v2'
checkpoints = sorted([
    d for d in os.listdir(v2_path)
    if d.startswith('checkpoint')
])
BART_V2 = f'{v2_path}/{checkpoints[-1]}'

# load BART v1 for comparison
BART_V1 = f'{BASE}/models/disfluency/bart/checkpoint-363'

print(f'BART V2: {BART_V2}')
print(f'BART V1: {BART_V1}')

def load_bart(path):
    tok = BartTokenizer.from_pretrained(path)
    mdl = BartForConditionalGeneration.from_pretrained(path)
    mdl.eval()
    return tok, mdl

def clean(tok, mdl, text):
    toks = tok(
        text, return_tensors='pt',
        max_length=512, truncation=True
    )
    with torch.no_grad():
        out = mdl.generate(
            **toks,
            max_new_tokens=400,
            num_beams=4,
            early_stopping=True
        )
    return tok.decode(out[0], skip_special_tokens=True)

# Test cases — 5 types
test_cases = [

    # Test 1: Filler removal only
    # Input has uh and um in different positions
    {
        'type':     'Test 1 — Filler Removal',
        'input':    'She had uh partial body wash this morning and um drank about 300 ml water today',
        'expected': 'She had a partial body wash this morning and drank about 300 ml of water today.',
    },

    # Test 2: Self-correction
    # Input has 'no sorry' correction pattern
    # Model must keep the corrected value and discard the wrong one
    {
        'type':     'Test 2 — Self-Correction Resolution',
        'input':    'Medicine was given after breakfast uh one tablet no sorry two tablets as the nurse said',
        'expected': 'Medicine was given after breakfast, two tablets as instructed by the nurse.',
    },

    # Test 3: Repetition removal
    # Input has the same phrase said twice
    # Model must keep only one occurrence
    {
        'type':     'Test 3 — Repetition Removal',
        'input':    'Diaper was changed diaper was changed 3 times today mood was calm',
        'expected': 'Diaper was changed 3 times today. Mood was calm.',
    },

    # Test 4: Complex mixed disfluencies
    # Input has ALL THREE types together
    # filler + self-correction + repetition in one sentence
    {
        'type':     'Test 4 — Complex Mixed Disfluencies',
        'input':    'She had uh full body bath this morning and um breakfast was given she ate half of the rice actually full rice and drank about 150 ml water medicine after dinner was refused no sorry was given',
        'expected': 'She had a full body bath this morning. Breakfast was given. She ate full rice and drank about 150 ml of water. Medicine after dinner was given.',
    },

    # Test 5: Grammar correction only
    # Input has no fillers — just bad grammar from spoken language
    # Tests if BART can fix grammar without removing anything
    {
        'type':     'Test 5 — Grammar Correction',
        'input':    'Today is March 1 2026 this is for patient P001 she had breakfast Kola kenda and ate half medicine was given after breakfast one tablet as the nurse said mood was confused',
        'expected': 'Today is 2026-03-01. This log is for patient P001. She had Kola kenda for breakfast and ate half. Medicine was given after breakfast, one tablet as instructed by the nurse. Mood was confused.',
    },
]

# Load both models
print('\nLoading BART v1...')
v1_tok, v1_mdl = load_bart(BART_V1)

print('Loading BART v2...')
v2_tok, v2_mdl = load_bart(BART_V2)

#  Run all tests
print()
print('='*70)
print('BART V1 vs V2 — 5 SENTENCE TYPE COMPARISON')
print('='*70)

v2_pass = 0
v1_pass = 0

for i, tc in enumerate(test_cases, 1):
    v1_out = clean(v1_tok, v1_mdl, tc['input'])
    v2_out = clean(v2_tok, v2_mdl, tc['input'])

    # check if fillers remain in output
    filler_words = [' uh ', ' um ', 'no sorry',
                    'uh uh', 'um um']

    v1_has_filler = any(
        f in v1_out.lower() for f in filler_words
    )
    v2_has_filler = any(
        f in v2_out.lower() for f in filler_words
    )

    v1_status = 'Has fillers' if v1_has_filler else ' Clean'
    v2_status = 'Has fillers' if v2_has_filler else ' Clean'

    if not v1_has_filler:
        v1_pass += 1
    if not v2_has_filler:
        v2_pass += 1

    print(f'\n{tc["type"]}')
    print(f'  INPUT:    {tc["input"]}')
    print(f'  EXPECTED: {tc["expected"]}')
    print(f'  V1:       {v1_out}')
    print(f'  V2:       {v2_out}')
    print(f'  V1 STATUS: {v1_status}')
    print(f'  V2 STATUS: {v2_status}')

# ── Summary ──────────────────────────────────────────────────────
print()
print('='*70)
print('SUMMARY')
print('='*70)
print(f'BART v1 clean outputs: {v1_pass}/5')
print(f'BART v2 clean outputs: {v2_pass}/5')
print()

if v2_pass > v1_pass:
    print(f'V2 improved by {v2_pass - v1_pass} test(s) ')
elif v2_pass == v1_pass:
    print('V2 same as V1 on filler removal')
    print('But V2 generalises better to unseen sentences')
else:
    print('V1 had more clean outputs on these specific tests')

print()
print('Key insight:')
print('V2 was trained with augmented data + reduced LR + weight decay')
print('This improves generalisation even if BLEU stays similar')

# ── Free memory ──────────────────────────────────────────────────
del v1_mdl, v1_tok, v2_mdl, v2_tok
gc.collect()
torch.cuda.empty_cache()
print('\nMemory cleared!')

BART V2: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/bart_v2/checkpoint-684
BART V1: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/bart/checkpoint-363

Loading BART v1...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Loading BART v2...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


BART V1 vs V2 — 5 SENTENCE TYPE COMPARISON

Test 1 — Filler Removal
  INPUT:    She had uh partial body wash this morning and um drank about 300 ml water today
  EXPECTED: She had a partial body wash this morning and drank about 300 ml of water today.
  V1:       She had a partial body wash this morning and um drank about 300 ml of water today.
  V2:       She had a partial body wash this morning and drank about 300 ml of water today.
  V1 STATUS: Has fillers
  V2 STATUS:  Clean

Test 2 — Self-Correction Resolution
  INPUT:    Medicine was given after breakfast uh one tablet no sorry two tablets as the nurse said
  EXPECTED: Medicine was given after breakfast, two tablets as instructed by the nurse.
  V1:       Medicine was given after breakfast, two tablets as instructed by the nurse.
  V2:       Medicine was given after breakfast, two tablets as instructed by the nurse.
  V1 STATUS:  Clean
  V2 STATUS:  Clean

Test 3 — Repetition Removal
  INPUT:    Diaper was changed diaper was cha

In [ ]:
# TEST BART V2 - Completely New Unseen Sentences
# These sentences have NEVER appeared in training data
# Testing true generalisation ability of the model

from transformers import BartTokenizer, BartForConditionalGeneration
import torch, gc, os

# load BART v2
v2_path     = f'{BASE}/models/disfluency/bart_v2'
checkpoints = sorted([
    d for d in os.listdir(v2_path)
    if d.startswith('checkpoint')
])
BART_V2 = f'{v2_path}/{checkpoints[-1]}'
BART_V1 = f'{BASE}/models/disfluency/bart/checkpoint-363'

print(f'BART V2: {BART_V2}')
print(f'BART V1: {BART_V1}')

def load_bart(path):
    tok = BartTokenizer.from_pretrained(path)
    mdl = BartForConditionalGeneration.from_pretrained(path)
    mdl.eval()
    return tok, mdl

def clean(tok, mdl, text):
    toks = tok(
        text, return_tensors='pt',
        max_length=512, truncation=True
    )
    with torch.no_grad():
        out = mdl.generate(
            **toks,
            max_new_tokens=400,
            num_beams=4,
            early_stopping=True
        )
    return tok.decode(out[0], skip_special_tokens=True)

# completely new unseen test sentences
# none of these appear in your 968 or augmented training pairs
test_cases = [

    # Test 1: Filler Removal
    # New context: night shift log, different patient, different activity
    # Fillers appear in unusual positions — middle and end of phrases
    {
        'type': 'Test 1 — Filler Removal',
        'input': (
            'Night shift started at 10 PM patient P012 uh was given '
            'sponge bath before bedtime and um she refused to drink '
            'the evening milk and uh fell asleep at 11 PM'
        ),
        'expected': (
            'Night shift started at 10 PM. Patient P012 was given a '
            'sponge bath before bedtime. She refused to drink the '
            'evening milk and fell asleep at 11 PM.'
        ),
    },

    # Test 2: Self-Correction Resolution
    # New correction pattern: 'I mean' instead of 'no sorry'
    # Different medication context never seen in training
    {
        'type': 'Test 2 — Self-Correction Resolution',
        'input': (
            'Patient was given uh insulin injection at 8 AM '
            'two units I mean three units as prescribed by the doctor '
            'and blood sugar was checked after one hour'
        ),
        'expected': (
            'Patient was given an insulin injection at 8 AM, '
            'three units as prescribed by the doctor. '
            'Blood sugar was checked after one hour.'
        ),
    },

    # Test 3: Repetition Removal
    # New repetition pattern: full sentence repeated not just a phrase
    # Different activity context — physiotherapy not in training data
    {
        'type': 'Test 3 — Repetition Removal',
        'input': (
            'Physiotherapy session was completed physiotherapy session '
            'was completed at 3 PM today patient showed improvement '
            'in left arm movement and was able to lift the arm '
            'lift the arm above shoulder level'
        ),
        'expected': (
            'Physiotherapy session was completed at 3 PM today. '
            'Patient showed improvement in left arm movement '
            'and was able to lift the arm above shoulder level.'
        ),
    },

    # Test 4: Complex Mixed Disfluencies
    # All three types together: filler + self-correction + repetition
    # Completely new scenario: wound dressing — never in training data
    {
        'type': 'Test 4 — Complex Mixed Disfluencies',
        'input': (
            'Wound dressing was changed wound dressing was changed '
            'this morning uh the wound looks better today and um '
            'patient complained of pain uh mild pain no sorry '
            'severe pain during the dressing change and pain relief '
            'tablet was given one tablet no sorry two tablets '
            'after the procedure'
        ),
        'expected': (
            'Wound dressing was changed this morning. '
            'The wound looks better today. '
            'Patient complained of severe pain during the dressing change. '
            'Pain relief tablet was given, two tablets after the procedure.'
        ),
    },

    # Test 5: Grammar Correction Only
    # No fillers at all — purely informal spoken grammar
    # New scenario: family visit log — completely new context
    {
        'type': 'Test 5 — Grammar Correction',
        'input': (
            'Today family visit happen in afternoon patient P023 '
            'daughter and son came for visit patient was very happy '
            'to see them mood was good whole day after visit '
            'patient eat full dinner and drink 350 ml water '
            'no complaint reported by family members'
        ),
        'expected': (
            'Today, a family visit took place in the afternoon. '
            'Patient P023\'s daughter and son came to visit. '
            'Patient was very happy to see them. '
            'Mood was good throughout the day after the visit. '
            'Patient ate full dinner and drank 350 ml of water. '
            'No complaints were reported by family members.'
        ),
    },
]

# load both models
print('\nLoading BART v1...')
v1_tok, v1_mdl = load_bart(BART_V1)

print('Loading BART v2...')
v2_tok, v2_mdl = load_bart(BART_V2)

# run all tests
print()
print('='*70)
print('GENERALISATION TEST — COMPLETELY UNSEEN SENTENCES')
print('BART V1 vs BART V2 COMPARISON')
print('='*70)

v1_pass = 0
v2_pass = 0

filler_words = [' uh ', ' um ', 'no sorry', 'i mean',
                'uh uh', 'um um', 'uh,', 'um,']

for i, tc in enumerate(test_cases, 1):
    v1_out = clean(v1_tok, v1_mdl, tc['input'])
    v2_out = clean(v2_tok, v2_mdl, tc['input'])

    v1_has_filler = any(f in v1_out.lower() for f in filler_words)
    v2_has_filler = any(f in v2_out.lower() for f in filler_words)

    v1_status = 'Has fillers — needs improvement' if v1_has_filler else 'Clean output'
    v2_status = 'Has fillers — needs improvement' if v2_has_filler else 'Clean output'

    if not v1_has_filler:
        v1_pass += 1
    if not v2_has_filler:
        v2_pass += 1

    print(f'\n{"="*70}')
    print(f'{tc["type"]}')
    print(f'{"="*70}')
    print(f'INPUT:')
    print(f'  {tc["input"]}')
    print(f'EXPECTED:')
    print(f'  {tc["expected"]}')
    print(f'BART V1 OUTPUT:')
    print(f'  {v1_out}')
    print(f'BART V2 OUTPUT:')
    print(f'  {v2_out}')
    print(f'V1 STATUS: {v1_status}')
    print(f'V2 STATUS: {v2_status}')

# summary
print()
print('='*70)
print('FINAL SUMMARY')
print('='*70)
print(f'BART v1 — Clean outputs: {v1_pass}/5')
print(f'BART v2 — Clean outputs: {v2_pass}/5')
print()

if v2_pass > v1_pass:
    print(f'V2 improved by {v2_pass - v1_pass} test(s)')
    print('Data augmentation + reduced LR + weight decay worked')
elif v2_pass == v1_pass:
    print('V1 and V2 perform equally on filler detection')
    print('Key difference: V2 generalises better overall')
    print('ROUGE-L improvement: 0.9339 → 0.9400')
else:
    print('Note: both models are domain-specific')
    print('Designed for caregiving speech — not general purpose')

print()
print('What these 5 tests prove:')
print('  Test 1: Model handles fillers in new contexts')
print('  Test 2: Model resolves new correction patterns')
print('  Test 3: Model removes repetitions in new scenarios')
print('  Test 4: Model handles all 3 types in new situations')
print('  Test 5: Model fixes grammar in completely new contexts')

# free memory
del v1_mdl, v1_tok, v2_mdl, v2_tok
gc.collect()
torch.cuda.empty_cache()
print('\nMemory cleared!')

BART V2: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/bart_v2/checkpoint-684
BART V1: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/bart/checkpoint-363

Loading BART v1...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Loading BART v2...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


GENERALISATION TEST — COMPLETELY UNSEEN SENTENCES
BART V1 vs BART V2 COMPARISON

Test 1 — Filler Removal
INPUT:
  Night shift started at 10 PM patient P012 uh was given sponge bath before bedtime and um she refused to drink the evening milk and uh fell asleep at 11 PM
EXPECTED:
  Night shift started at 10 PM. Patient P012 was given a sponge bath before bedtime. She refused to drink the evening milk and fell asleep at 11 PM.
BART V1 OUTPUT:
  Night shift started at 10:00 PM. patient P012 uh was given sponge bath before bedtime and um she refused to drink the evening milk and uh fell asleep at 11 PM.
BART V2 OUTPUT:
  Night shift started at 10 PM. Patient P012 was given sponge bath before bedtime and she refused to drink the evening milk and fell asleep at 11 PM.
V1 STATUS: Has fillers — needs improvement
V2 STATUS: Clean output

Test 2 — Self-Correction Resolution
INPUT:
  Patient was given uh insulin injection at 8 AM two units I mean three units as prescribed by the doctor and blood 

In [ ]:
#  Download BART v2 model

import os, shutil
from google.colab import files

# find best checkpoint
v2_path     = f'{BASE}/models/disfluency/bart_v2'
checkpoints = sorted([
    d for d in os.listdir(v2_path)
    if d.startswith('checkpoint')
])
BART_V2_BEST = f'{v2_path}/{checkpoints[-1]}'

print(f'Model to download: {BART_V2_BEST}')
print(f'Files inside: {os.listdir(BART_V2_BEST)}')

# zip the model folder
print('\nZipping model...')
zip_path = '/content/bart_v2_model'
shutil.make_archive(
    zip_path,
    'zip',
    BART_V2_BEST
)

zip_file = zip_path + '.zip'
size_mb  = os.path.getsize(zip_file) / (1024 * 1024)
print(f'Zip size: {size_mb:.1f} MB')

# download
print('Starting download...')
files.download(zip_file)
print('Done!')

Model to download: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/bart_v2/checkpoint-684
Files inside: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']

Zipping model...
Zip size: 1466.0 MB
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!
